## Sequential Panorama Stitching

# Building the Sequential Multi-Image Panorama Pipeline

Welcome back! In our first unit, you learned how to merge two aligned image canvases into a single smooth panorama. In the second unit, you learned how to combine all the complex steps of finding features, matching them, and generating a diagnostic report into a single, reliable `stitch_pair` function.

Now, we are ready to build the complete, final panorama pipeline.

A large panorama is built sequentially. We start with a base image (the first picture), stitch the next image to it, and then use that newly combined result as the new base for the third image, and so on.

However, **sequential stitching introduces a significant challenge: error accumulation**. When we stitch multiple images in a row, a tiny mistake on the second image will make the fourth image look terrible or cause it to fail entirely. A later failure is often caused by an earlier, imperfect stitch. This is precisely why our script needs to print diagnostics after every single step, allowing us to inspect the process as it happens.

---

## Formatting the Step Report

To understand what our pipeline is doing, we first need a small helper function. In our previous unit, you saw that our `stitch_pair` function returns a dictionary called `report` containing helpful data, such as the number of successful matches and inliers.

We need to print this data to the screen in a readable way. In Python, you can use the `.items()` method on a dictionary to loop through all its keys and values simultaneously.

```python
def print_step_report(index, report):
    print(f"image {index}:")
    for key, value in report.items():
        print(f"  {key}: {value}")

```

In this code, `index` tells us which image in the sequence we just processed. By calling `report.items()`, the `for` loop iterates through every piece of data inside the report one by one. The `key` might be a word such as `"matches"`, and the `value` would be a number such as `150`.

If we pass an image index of `2` and a small report dictionary into this function, the output on the screen will look like this:

```text
image 2:
  status: ok
  matches: 150
  inliers: 120

```

This ensures we always know exactly what is happening under the hood after every stitch.

---

## The Sequential Stitching Loop

Now we can move to the core logic of our script. Imagine we have a list of image paths provided by the user, stored in `args.paths`.

We first load the very first image in that list and save it to a variable called `panorama`. This is our starting canvas. Next, we loop over the remaining images in the list.

```python
# We skip the first image using [1:] because it is already our base panorama
for index, path in enumerate(args.paths[1:], start=2):
    next_image = read_color(path)

```

Here, `enumerate` is a helpful Python tool that provides both an index (a counting number) and the item itself (the `path`). We use `start=2` because we are looking at the second image in our sequence. Inside the loop, we read the image file from the path.

Next, we pass our current `panorama` and the `next_image` into our `stitch_pair` function from the previous unit.

```python
for index, path in enumerate(args.paths[1:], start=2):
    next_image = read_color(path)
    stitched, report = stitch_pair(
        panorama,
        next_image,
        method=args.method,
        ratio=args.ratio,
        ransac_threshold=args.ransac_threshold,
    )
    print_step_report(index, report)

```

After the function finishes, we immediately call the `print_step_report` function we just built to print the results of that specific stitch.

Finally, we must update our `panorama` variable. By setting `panorama` equal to the newly stitched image, the next run of the loop will use this updated, wider canvas as the base for the third image.

```python
for index, path in enumerate(args.paths[1:], start=2):
    next_image = read_color(path)
    stitched, report = stitch_pair(
        panorama,
        next_image,
        method=args.method,
        ratio=args.ratio,
        ransac_threshold=args.ransac_threshold,
    )
    print_step_report(index, report)
    panorama = stitched

```

---

## Handling Failures and Warnings

Our loop looks great, but it lacks a crucial safety feature. If two images do not share enough common features, the `stitch_pair` function will fail and return `None` instead of a stitched image. If we try to use `None` as our base panorama in the next loop, our script will crash with an unhandled error.

To fix this, we need to add safety checks inside our loop before we update the `panorama` variable.

```python
        print_step_report(index, report)
        if stitched is None:
            raise SystemExit("sequence stopped; inspect the failed pair before continuing")
        if report["inlier_ratio"] < args.min_inlier_ratio:
            print("  warning: low inlier ratio; inspect this pair before trusting the panorama")
        panorama = stitched

```

* **Handling Missing Stitches:** We use an `if` statement to check if `stitched` is `None`. If it is, `raise SystemExit(...)` safely stops the program immediately and outputs an explanation message.
* **Low Inlier Ratio Warning:** We check `report["inlier_ratio"]`. If this ratio falls below the minimum threshold (e.g., `0.25` or 25%), we log a warning so the user knows this pair may introduce distortion or misalignment into the final composition.

---

## Course Wrap-Up and Final Practices

Congratulations! You have reached the final lesson of the course. Building a computer vision pipeline from scratch is no easy task, but you have successfully learned how to process images, find matching points, stitch them together mathematically, and build a robust, safe program to automate the process for entire sequences of photos.

In this lesson, we combined the powerful tools from our previous units with a sequential loop and crucial safety checks. By printing a report after every step and stopping the program safely upon failure, you have created a truly professional-grade panorama generator.

In the upcoming practice exercises, you will put everything together. You will write the loop and reporting logic we just discussed, test your completed script on a provided sequence of sample images to observe how the terminal output updates step-by-step, and finally, upload your very own custom photos to create a personal panorama. Let's get started on the final challenge.

## Reporting Every Stitch Step

Now that your stitch_pair function hands back a detailed report after every merge, it is time to make that report easy to read on screen.

In this first exercise of the unit, your job is to finish the print_step_report helper. This small function takes an image index and a report dictionary and turns them into clean, readable lines in the terminal.

Inside the function:

    Print a header line in the format image {index}:.

    Loop over report.items() and print each key/value pair on its own line, indented with two spaces, like {key}: {value}.

Getting this helper right means every step of your panorama pipeline will give you clear feedback, which is a great foundation for the loop you'll build next.

```
import argparse
import cv2

from cvkit import read_color
from stitching import stitch_pair


# TODO: This helper prints the report dictionary returned by stitch_pair.
def print_step_report(index, report):
    # TODO: Print the header line in the format "image {index}:".

    # TODO: Loop over report.items() and print each pair on its own line,
    #       indented with two spaces, like "  {key}: {value}".

    pass


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("paths", nargs="+")
    parser.add_argument("--out", default="panorama.jpg")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    parser.add_argument("--min-inlier-ratio", type=float, default=0.25)
    args = parser.parse_args()

    if len(args.paths) < 2:
        raise SystemExit("Provide at least two overlapping images")

    panorama = read_color(args.paths[0])

    cv2.imwrite(args.out, panorama)
    cv2.imshow("panorama", panorama)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
```

Here is the completed implementation of `print_step_report` in `solution.py`:

```python
import argparse
import cv2

from cvkit import read_color
from stitching import stitch_pair


def print_step_report(index, report):
    # 1. Print the header line
    print(f"image {index}:")

    # 2. Print each key/value pair indented with two spaces
    for key, value in report.items():
        print(f"  {key}: {value}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("paths", nargs="+")
    parser.add_argument("--out", default="panorama.jpg")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    parser.add_argument("--min-inlier-ratio", type=float, default=0.25)
    args = parser.parse_args()

    if len(args.paths) < 2:
        raise SystemExit("Provide at least two overlapping images")

    panorama = read_color(args.paths[0])

    cv2.imwrite(args.out, panorama)
    cv2.imshow("panorama", panorama)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```

## Building the Stitching Loop

Now that your print_step_report helper prints clean feedback for each step, it is time to put it to work inside the heart of the pipeline: the sequential stitching loop.

Your task is to complete the loop that iterates through every image after the base one and stitches them together, one at a time. Look for the TODO block inside main() and build the loop using enumerate(args.paths[1:], start=2).

Inside the loop, you will:

    Read the next image from path using read_color(...).
    Call stitch_pair(...) with the current panorama and the next image, passing method=args.method, ratio=args.ratio, and ransac_threshold=args.ransac_threshold, and then unpack the result into stitched, report.
    Print the report using print_step_report(index, report).
    Terminate the program with raise SystemExit("sequence stopped; inspect the failed pair before continuing") if stitched is None.
    Print a warning if report["inlier_ratio"] < args.min_inlier_ratio.
    Update panorama = stitched so that the next pass builds on the fresh canvas.

This loop transforms a pair of photos into a full panorama; completing it brings the entire course together.

```
import argparse
import cv2

from cvkit import read_color
from stitching import stitch_pair


def print_step_report(index, report):
    print(f"image {index}:")
    for key, value in report.items():
        print(f"  {key}: {value}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("paths", nargs="+")
    parser.add_argument("--out", default="panorama.jpg")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    parser.add_argument("--min-inlier-ratio", type=float, default=0.25)
    args = parser.parse_args()

    if len(args.paths) < 2:
        raise SystemExit("Provide at least two overlapping images")

    panorama = read_color(args.paths[0])

    # TODO: Build the sequential stitching loop here.
    #
    #       Loop over the remaining images with
    #       enumerate(args.paths[1:], start=2). Each loop gives you an
    #       `index` (which starts at 2) and a `path`.
    #
    #       Inside the loop:
    #       1. Read the next image from `path` with read_color(...).
    #       2. Call stitch_pair(...) with the current `panorama` and the next
    #          image, passing method=args.method, ratio=args.ratio, and
    #          ransac_threshold=args.ransac_threshold. Unpack the returned
    #          values into `stitched, report`.
    #       3. Print the report with print_step_report(index, report).
    #       4. If `stitched is None`, stop the program with
    #          raise SystemExit("sequence stopped; inspect the failed pair before continuing").
    #       5. If report["inlier_ratio"] < args.min_inlier_ratio, print a
    #          warning so the user can inspect this pair.
    #       6. Update `panorama = stitched` so the next loop builds on the
    #          newly stitched canvas.

    cv2.imwrite(args.out, panorama)
    cv2.imshow("panorama", panorama)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
```

Here is the completed implementation for `solution.py`:

```python
import argparse
import cv2

from cvkit import read_color
from stitching import stitch_pair


def print_step_report(index, report):
    print(f"image {index}:")
    for key, value in report.items():
        print(f"  {key}: {value}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("paths", nargs="+")
    parser.add_argument("--out", default="panorama.jpg")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    parser.add_argument("--min-inlier-ratio", type=float, default=0.25)
    args = parser.parse_args()

    if len(args.paths) < 2:
        raise SystemExit("Provide at least two overlapping images")

    panorama = read_color(args.paths[0])

    # Sequential stitching loop
    for index, path in enumerate(args.paths[1:], start=2):
        # 1. Read the next image
        next_image = read_color(path)

        # 2. Stitch current panorama and next image
        stitched, report = stitch_pair(
            panorama,
            next_image,
            method=args.method,
            ratio=args.ratio,
            ransac_threshold=args.ransac_threshold,
        )

        # 3. Print step report
        print_step_report(index, report)

        # 4. Stop if stitching failed
        if stitched is None:
            raise SystemExit("sequence stopped; inspect the failed pair before continuing")

        # 5. Check inlier ratio threshold warning
        if report["inlier_ratio"] < args.min_inlier_ratio:
            print("  warning: low inlier ratio; inspect this pair before trusting the panorama")

        # 6. Accumulate panorama canvas
        panorama = stitched

    cv2.imwrite(args.out, panorama)
    cv2.imshow("panorama", panorama)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```

## Watch Your Panorama Come Together

The loop you assembled in the previous exercise is now part of a complete, end-to-end panorama pipeline that is ready to run from start to finish.

This time, there is no new code to write. Your task is to run the completed script and observe the entire system functioning as a single unit.

Point it at a sequence of overlapping images, like so: python3 solution.py sample_images/building/1.jpg sample_images/building/2.jpg sample_images/building/3.jpg. You can even try to upload your own photos!

As it runs, read the per-step report that is displayed after every stitch, then open the saved panorama.jpg and examine how the images align.

Observing the reports and the result side by side is the best way to understand how every component you built integrates into a single working tool.

```
import argparse
import cv2

from cvkit import read_color
from stitching import stitch_pair


# This is the complete panorama pipeline you built across this lesson.
# There is nothing to write here. Your job is to RUN this script and watch it
# work: read the per-step report that prints after every stitch, and inspect
# the final panorama it produces.
def print_step_report(index, report):
    print(f"image {index}:")
    for key, value in report.items():
        print(f"  {key}: {value}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("paths", nargs="+")
    parser.add_argument("--out", default="panorama.jpg")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    parser.add_argument("--min-inlier-ratio", type=float, default=0.25)
    args = parser.parse_args()

    if len(args.paths) < 2:
        raise SystemExit("Provide at least two overlapping images")

    panorama = read_color(args.paths[0])

    for index, path in enumerate(args.paths[1:], start=2):
        next_image = read_color(path)

        stitched, report = stitch_pair(
            panorama,
            next_image,
            method=args.method,
            ratio=args.ratio,
            ransac_threshold=args.ransac_threshold,
        )

        print_step_report(index, report)

        if stitched is None:
            raise SystemExit("sequence stopped; inspect the failed pair before continuing")

        if report["inlier_ratio"] < args.min_inlier_ratio:
            print("  warning: low inlier ratio; inspect this pair before trusting the panorama")

        panorama = stitched

    cv2.imwrite(args.out, panorama)
    cv2.imshow("panorama", panorama)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```